# Détection d’erreurs phonétiques en arabe

# Problématique

Dans le domaine de l’orthophonie, l’évaluation et la prise en charge des troubles de la parole en langue arabe constituent un défi majeur. Cette complexité s’explique notamment par la richesse phonétique de l’arabe, caractérisée par la présence de phonèmes spécifiques (tels que les consonnes emphatiques), ainsi que par la variabilité importante des profils des patients.

En pratique, les orthophonistes doivent analyser avec précision les productions orales afin d’identifier les erreurs phonétiques. Ce processus repose essentiellement sur une expertise humaine, ce qui le rend à la fois chronophage et potentiellement subjectif. De plus, le manque d’outils technologiques adaptés à la langue arabe limite fortement les possibilités d’automatisation et d’assistance au diagnostic.

Dans ce contexte, l’intégration des techniques de Machine Learning offre une perspective prometteuse pour automatiser la détection des erreurs de prononciation. Toutefois, plusieurs défis subsistent, notamment :
- la rareté des bases de données audio annotées  
- la variabilité inter-individuelle des troubles du langage  
- la nécessité de concevoir des modèles fiables et interprétables

# Dataset
Le dataset Mozilla Common Voice 18 (édition arabe) est une extraction non officielle du corpus Common Voice, dédiée exclusivement à la langue arabe. Il est conçu pour les travaux de recherche et de développement en reconnaissance automatique de la parole (ASR).

Ce jeu de données conserve la structure originale du corpus tout en filtrant uniquement les enregistrements en arabe. Il inclut des fichiers audio accompagnés de leurs transcriptions ainsi que des métadonnées sur les locuteurs.
Chaque exemple du dataset contient :

* Un enregistrement audio en arabe (échantillonné à 48 kHz)
* Une transcription textuelle correspondante
Des informations sur le locuteur :
âge / genre / accent /Localisation (locale) /Des données de validation basées sur des votes (positifs et négatifs)

## Structure des données

Les principales caractéristiques des données sont :

client_id : identifiant anonyme du locuteur<br>
audio : signal audio<br>
sentence : transcription en arabe<br>
up_votes / down_votes : validation par la communauté<br>
age, gender, accent, locale : métadonnées du locuteur<br>

## Le dataset est divisé en plusieurs sous-ensembles :
| Split       | Nombre d’exemples | Taille approximative |
| ----------- | ----------------- | -------------------- |
| Train       | 28,410            | ~766 MB              |
| Validation  | 10,471            | ~313 MB              |
| Test        | 10,471            | ~324 MB              |
| Other       | 41,586            | ~1.28 GB             |
| Invalidated | 15,120            | ~497 MB              |
| **Total**   | **105,058**       | **~3.18 GB**         |

Train : utilisé pour entraîner le modèle et lui apprendre à reconnaître la parole.

Validation : utilisé pour vérifier et ajuster le modèle pendant l’entraînement.

Test : utilisé pour évaluer les performances finales du modèle sur des données jamais vues.

Other : contient des données non validées dont la qualité n’est pas garantie.

Invalidated : contient des données rejetées car incorrectes ou de mauvaise qualité.

Source : https://huggingface.co/datasets/MohamedRashad/common-voice-18-arabic



# Etat de l’art

La détection des erreurs de prononciation, connue sous le nom de Computer-Assisted Pronunciation Training (CAPT), s’appuie sur plusieurs approches issues de la reconnaissance automatique de la parole (Automatic Speech Recognition – ASR) et du traitement du signal audio.

Ces approches peuvent être classées en méthodes de deep learning et en méthodes d’apprentissage automatique classique.
# Méthodes existantes


# CNN et KNN

### le MFCC ?

Le MFCC est une technique utilisée pour représenter un signal audio sous forme de données numériques.

Il est très utilisé dans :

La reconnaissance vocale, l'analyse de la parole et le systèmes de transcription audio

### Pourquoi l’utiliser ?
Un signal audio brut :

est un signal complexe (ondes continues)<br>
contient beaucoup de bruit et d’informations inutiles<br>
est difficile à utiliser directement par un modèle d’IA<br>

=> Le MFCC permet de transformer ce signal en une forme plus simple et informative.
### LE CNN ?
Un CNN est un type de réseau de neurones conçu pour analyser des données structurées

### Pourquoi utiliser un CNN ici ?

Le MFCC peut être vu comme une image 2D (temps × fréquence).

=> Donc un CNN est adapté pour détecter des motifs dans cette matrice

Le CNN ne fait PAS directement de classification Il est utilisé comme :<br>
extracteur de caractéristiques (feature extractor)

### Fonctionnement du CNN
1. Convolution

Le CNN applique des filtres pour détecter des motifs dans le signal :

variations de fréquence<br>
patterns de prononciation<br>
structures acoustiques<br>

2. Pooling

Réduction de la taille des données pour :

garder les informations importantes<br>
réduire la complexité<br>

3. Embedding (représentation finale)

À la fin du CNN, on obtient :

Vecteur de 64 dimensions

=>Ce vecteur représente toute la prononciation de l’audio.

### KNN
K-Nearest Neighbors

KNN est un algorithme qui fonctionne en trouvant les exemples les plus proches d’un point donné.
### Principe de fonctionnement
Pour un nouvel audio :

on calcule son embedding<br>
on cherche les K embeddings les plus proches dans la base<br>
on analyse leurs étiquettes ou leurs similarités<br>

### Rôle dans ce projet
le KNN sert a :
Analyser les cas proches de l’utilisateur<br>
aider à comprendre la décision du modèle<br>

INSTALL DEPENDENCIES

In [ ]:
!pip install librosa datasets scikit-learn tensorflow soundfile

IMPORT LIBRARIES

In [ ]:
import numpy as np
import librosa
import random
import tensorflow as tf

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import files

Load dataset

In [ ]:
dataset = load_dataset(
    "MohamedRashad/common-voice-18-arabic",
    split="train"
)

MFCC extraction

In [ ]:
SR = 16000
N_MFCC = 40
MAX_LEN = 200

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC).T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

Prepare dataset

In [ ]:
X = []

for sample in dataset.select(range(1000)):
    try:
        audio = sample["audio"]["array"]
        mfcc = extract_mfcc(audio)
        X.append(mfcc)
    except:
        continue

X = np.array(X)
print("Dataset shape:", X.shape)

Dataset shape: (1000, 200, 40)


Train

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

CNN

In [ ]:
inputs = tf.keras.Input(shape=(200, 40))

# Encoder
x = tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling1D(2, padding='same')(x)

x = tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPooling1D(2, padding='same')(x)

x = tf.keras.layers.Flatten()(x)
embedding = tf.keras.layers.Dense(64, activation='relu', name="embedding")(x)

# Decoder
x = tf.keras.layers.Dense(50 * 128, activation='relu')(embedding)
x = tf.keras.layers.Reshape((50, 128))(x)

x = tf.keras.layers.UpSampling1D(2)(x)
x = tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same')(x)

x = tf.keras.layers.UpSampling1D(2)(x)
outputs = tf.keras.layers.Conv1D(40, 3, activation='linear', padding='same')(x)

model = tf.keras.Model(inputs, outputs)

Train CNN with dummy target

In [ ]:
model.compile(optimizer='adam', loss='mse')

model.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - loss: 2385.7124 - val_loss: 641.8176
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 561.9817 - val_loss: 459.7875
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - loss: 446.6687 - val_loss: 400.2577
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 418.5769 - val_loss: 388.3857
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 409.4290 - val_loss: 379.3008
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 389.4238 - val_loss: 348.1679
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 351.0427 - val_loss: 319.1500
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 314.7576 - val_loss: 292.5998
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 294.1312 - val_loss: 278.2731
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - loss: 281.7542 - val_loss: 269.7230


Extract embeddings

In [ ]:
encoder = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output
)

Build embedding dataset

In [ ]:
X_train_emb = encoder.predict(X_train)

print("Embedding shape:", X_train_emb.shape)  # should be (N, 64)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Embedding shape: (800, 64)


Train KNN

In [ ]:
nn = NearestNeighbors(n_neighbors=5, metric='cosine')
nn.fit(X_train_emb)

NearestNeighbors(metric='cosine')

Pick sentence

In [ ]:
sample = random.choice(list(dataset))

sentence = sample["sentence"]
reference_audio = sample["audio"]["array"]

print("🎤 اقرأ هذه الجملة:")
print("👉", sentence)

🎤 اقرأ هذه الجملة:
👉 كانت رخيصة الثمن جداً.


extract reference embedding

In [ ]:
ref_mfcc = extract_mfcc(reference_audio)
ref_emb = encoder.predict(np.expand_dims(ref_mfcc, axis=0))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step


Upload Audio

In [ ]:
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

Saving Enregistrement (39).m4a to Enregistrement (39).m4a


Process your audio

In [ ]:
user_audio, _ = librosa.load(audio_path, sr=SR)
user_mfcc = extract_mfcc(user_audio)

user_emb = encoder.predict(np.expand_dims(user_mfcc, axis=0))

/tmp/ipykernel_6523/2112413200.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  user_audio, _ = librosa.load(audio_path, sr=SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step


Main similarity

In [ ]:
score = cosine_similarity(user_emb, ref_emb)[0][0]

print("\n🎯 Similarity with reference:", score)


🎯 Similarity with reference: 0.50634986


KNN Analysis

In [ ]:
distances, indices = nn.kneighbors(user_emb)

print("\nNearest distances:", distances)


Nearest distances: [[0.01471376 0.01771963 0.02478302 0.03538501 0.04263586]]


Final decision

In [ ]:
print(f"\n🎯 Similarity score: {score:.3f}")

if score > 0.85:
    print("✔ Good pronunciation")
elif score > 0.65:
    print("⚠️ Acceptable but not perfect")
else:
    print("❌ Incorrect pronunciation")


🎯 Similarity score: 0.506
❌ Incorrect pronunciation


# Random Forest

## Le Random Forest

Le Random Forest est un algorithme de machine learning supervisé basé sur un ensemble d’arbres de décision.

Il est utilisé principalement pour des tâches de classification ou de régression.

## Fonctionnement du Random Forest

### 1. Arbres de décision

Chaque arbre de décision apprend des règles simples à partir des données d’entrée.

Exemple :
- si distance faible → probablement correct  
- si distance élevée → probablement incorrect  

### 2. Ensemble (Forest)

- Plusieurs arbres sont entraînés sur des sous-ensembles des données  
- Chaque arbre donne une prédiction indépendante  
- Les résultats sont combinés  

Le résultat final est obtenu par vote majoritaire.

### 3. Décision finale

Le modèle retourne :
- Correct  
- Incorrect  
- Probabilité (confidence)  

## Feature Engineering

Le Random Forest ne peut pas traiter directement un signal audio.

Il faut transformer l’audio en variables numériques appelées features.

### Features utilisées :
- moyenne des différences entre MFCC  
- écart-type des différences  
- valeur maximale des écarts  
- distance globale (norme L2)  

Ces features représentent le niveau de similarité entre deux signaux audio.

## Principe du système

Pour un nouvel audio :

1. Extraction du MFCC  
2. Comparaison avec un audio de référence  
3. Calcul des différences entre les deux MFCC  
4. Transformation en vecteur de features  
5. Passage au Random Forest  
6. Classification finale  

## Rôle du Random Forest dans ce projet

Le Random Forest sert à :

- apprendre à distinguer une prononciation correcte et incorrecte  
- prendre une décision basée sur les caractéristiques audio extraites  

## Rôle de la distance

Même si le modèle est basé sur des arbres de décision, la notion de distance reste importante.

Interprétation :
- faible distance → audio similaire → correct  
- forte distance → audio différent → incorrect  

## Limites du Random Forest

- ne comprend pas directement le langage  
- ne capture pas les phonèmes  
- dépend fortement de la qualité des features  
- sensible au bruit et aux variations d’enregistrement  
- ne traite pas la structure temporelle de l’audio  

## Résumé du pipeline

Audio utilisateur + Audio de référence  
Extraction MFCC  
Calcul des différences  
Création des features  
Random Forest  
Classification  
Correct / Incorrect  
Score de confiance  

## Conclusion

Dans ce système :

- le MFCC transforme l’audio en données numériques  
- les features mesurent la similarité entre les signaux  
- le Random Forest apprend à classifier la prononciation  

C’est une approche classique, robuste, mais limitée pour des tâches avancées de reconnaissance de prononciation

Install

In [ ]:
!pip install librosa datasets scikit-learn soundfile

Imports

In [ ]:
import numpy as np
import librosa
import random

from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

Load Dataset

In [ ]:
dataset = load_dataset(
    "MohamedRashad/common-voice-18-arabic",
    split="train"
)

MFCC Extraction

In [ ]:
SR = 16000
N_MFCC = 40
MAX_LEN = 200

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC).T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

Feature Engineering

In [ ]:
def extract_features(audio1, audio2):
    mfcc1 = extract_mfcc(audio1)
    mfcc2 = extract_mfcc(audio2)

    diff = np.abs(mfcc1 - mfcc2)

    return np.array([
        np.mean(diff),
        np.std(diff),
        np.max(diff),
        np.linalg.norm(mfcc1 - mfcc2)
    ])

Build REAL training data

In [ ]:
X = []
y = []

samples = list(dataset.select(range(1500)))

for i in range(len(samples)):
    try:
        s1 = samples[i]
        a1 = s1["audio"]["array"]

        # ✔ SAME sentence (different speaker)
        found = False
        for j in range(i+1, len(samples)):
            s2 = samples[j]
            if s1["sentence"] == s2["sentence"]:
                a2 = s2["audio"]["array"]

                X.append(extract_features(a1, a2))
                y.append(1)
                found = True
                break

        # ❌ DIFFERENT sentence
        s3 = random.choice(samples)
        a3 = s3["audio"]["array"]

        X.append(extract_features(a1, a3))
        y.append(0)

    except:
        continue

X = np.array(X)
y = np.array(y)

print("Dataset:", X.shape, y.shape)

Dataset: (1500, 4) (1500,)


Normalize Features

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Train Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

model.fit(X, y)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

TEST SYSTEM

In [ ]:
sample = random.choice(list(dataset))

print("🎤 Please read this sentence:")
print("👉", sample["sentence"])

ref_audio = sample["audio"]["array"]

🎤 Please read this sentence:
👉 وَإِنْ تَجْهَرْ بِالْقَوْلِ فَإِنَّهُ يَعْلَمُ السِّرَّ وَأَخْفَى


Upload audio

In [ ]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

Saving Enregistrement (44).m4a to Enregistrement (44).m4a


Process + Predict

In [ ]:
user_audio, _ = librosa.load(audio_path, sr=SR)

features = extract_features(user_audio, ref_audio)

# distance (IMPORTANT)
distance = features[-1] / (200 * 40)

features = scaler.transform(features.reshape(1, -1))

pred = model.predict(features)[0]
proba = model.predict_proba(features)[0]

confidence = max(proba)

print("\n🎯 Confidence:", confidence)
print("📏 Distance:", distance)

/tmp/ipykernel_3287/2175671060.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  user_audio, _ = librosa.load(audio_path, sr=SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



🎯 Confidence: 1.0
📏 Distance: 0.392223


FINAL DECISION

In [ ]:
if distance < 0.8:
    print("✔ Correct sentence")
elif distance < 1.2:
    print("⚠️ Acceptable")
else:
    print("❌ Incorrect sentence")

✔ Correct sentence


# Linear Regression

# La Régression Linéaire

La régression linéaire est un algorithme de machine learning supervisé utilisé pour prédire une valeur numérique continue.


## Pourquoi utiliser Linear Regression ici ?

Contrairement aux modèles de classification comme le Random Forest ou le SVM :

- elle ne prédit pas directement une classe (correct / incorrect)
- elle produit une valeur continue appelée score

Elle est donc utilisée pour :
- estimer le niveau de similarité entre deux signaux audio  
- transformer cette similarité en un score de prononciation  


# Fonctionnement de la Régression Linéaire

## 1. Modèle linéaire

Le modèle apprend une relation mathématique entre :

features → score


## 2. Apprentissage

Le modèle ajuste une fonction linéaire afin d’approximer :

- score proche de 1 → audios très similaires  
- score proche de 0 → audios très différents  


## 3. Décision finale

Le modèle retourne un score continu.

Ensuite, un seuil est appliqué pour la classification :

- score élevé → prononciation correcte  
- score moyen → prononciation acceptable  
- score faible → prononciation incorrecte  

# Feature Engineering

La régression linéaire ne peut pas traiter directement les signaux audio.

Il est donc nécessaire de transformer l’audio en variables numériques appelées features.


## Features utilisées

- moyenne des différences entre MFCC  
- écart-type des différences  
- distance globale (norme L2)  
- similarité cosinus  

Ces features représentent le niveau de similarité entre deux signaux audio.


# Principe du système

Pour un nouvel audio :

1. Extraction du MFCC  
2. Comparaison avec un audio de référence  
3. Calcul des différences entre les deux signaux  
4. Transformation en vecteur de features  
5. Passage au modèle de régression linéaire  
6. Génération d’un score de similarité  


# Rôle de la Régression Linéaire dans ce projet

La régression linéaire est utilisée pour :

- estimer un score de similarité entre deux audios  
- transformer des caractéristiques audio en valeur numérique exploitable  


# Interprétation du score

Le modèle produit un score continu représentant la similarité.

Interprétation :

- score élevé → prononciation correcte  
- score moyen → prononciation partiellement correcte  
- score faible → prononciation incorrecte  


# Limites de la Régression Linéaire

- modèle trop simple pour la complexité de la parole  
- ne comprend pas les mots ni les phonèmes  
- dépend fortement de la qualité des features  
- sensible au bruit et aux variations de voix  
- ne capture pas la structure temporelle du signal audio  


# Résumé du pipeline

Audio utilisateur + Audio de référence  
Extraction MFCC  
Calcul des différences  
Création des features  
Régression linéaire  
Score de similarité  
Décision finale  

# Conclusion

Dans ce système :

- le MFCC transforme le signal audio en représentation numérique  
- les features mesurent la similarité entre les deux audios  
- la régression linéaire calcule un score continu  
- ce score est utilisé pour estimer la qualité de la prononciation  

Install

In [1]:
!pip install librosa datasets scikit-learn soundfile

Imports

In [2]:
import numpy as np
import librosa
import random

from datasets import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

Load dataset

In [3]:
dataset = load_dataset(
    "MohamedRashad/common-voice-18-arabic",
    split="train"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/346M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/309M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/28410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10471 [00:00<?, ? examples/s]

Generating other split:   0%|          | 0/41586 [00:00<?, ? examples/s]

Generating invalidated split:   0%|          | 0/15120 [00:00<?, ? examples/s]

MFCC extraction

In [4]:
SR = 16000
N_MFCC = 40
MAX_LEN = 200

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC).T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

IMPROVED FEATURE ENGINEERING

In [5]:
def extract_features(audio1, audio2):
    mfcc1 = extract_mfcc(audio1)
    mfcc2 = extract_mfcc(audio2)

    diff = np.abs(mfcc1 - mfcc2)

    return np.array([
        np.mean(diff),
        np.std(diff),
        np.max(diff),
        np.linalg.norm(mfcc1 - mfcc2)
    ])

training data

In [13]:
X = []
y = []

samples = list(dataset.select(range(1500)))

for i in range(len(samples)):
    try:
        s1 = samples[i]
        a1 = s1["audio"]["array"]

        # ✔ SAME sentence (search for another speaker)
        for j in range(i+1, len(samples)):
            s2 = samples[j]

            if s1["sentence"] == s2["sentence"]:
                a2 = s2["audio"]["array"]

                f = extract_features(a1, a2)

                X.append(f)
                y.append(1.0)   # correct
                break

        # ❌ DIFFERENT sentence
        s3 = random.choice(samples)
        a3 = s3["audio"]["array"]

        f = extract_features(a1, a3)

        X.append(f)
        y.append(0.0)   # incorrect

    except:
        continue

X = np.array(X)
y = np.array(y)

print("Dataset:", X.shape)

Dataset: (1500, 4)


Normalize features

In [14]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Train Linear Regression

In [15]:
model = LinearRegression()
model.fit(X, y)

LinearRegression()

TEST SYSTEM

In [16]:
sample = random.choice(list(dataset))

print("🎤 Please read this sentence:")
print("👉", sample["sentence"])

ref_audio = sample["audio"]["array"]

🎤 Please read this sentence:
👉 ماذا؟ لم أسمعك.


Upload user audio

In [17]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

Saving Enregistrement (48).m4a to Enregistrement (48).m4a


Process audio

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

mfcc1 = extract_mfcc(user_audio).flatten().reshape(1, -1)
mfcc2 = extract_mfcc(ref_audio).flatten().reshape(1, -1)

score = cosine_similarity(mfcc1, mfcc2)[0][0]

print("\n🎯 Similarity Score:", score)


🎯 Similarity Score: 0.69857395


Result

In [21]:
if score > 0.6:
    print("✔ Correct pronunciation")
elif score > 0.4:
    print("⚠️ Acceptable")
else:
    print("❌ Incorrect")

✔ Correct pronunciation


# SVM (Support Vector Machine)

Le SVM est un algorithme de machine learning supervisé utilisé pour la classification.

Il permet de séparer des données en différentes classes en construisant une frontière optimale appelée hyperplan.


## Pourquoi utiliser le SVM ici ?

Dans ce projet, le SVM est utilisé pour :

- classer une prononciation en :
  - correcte  
  - incorrecte  

Il apprend à partir de caractéristiques audio (features) si deux signaux sont similaires ou différents.


# Fonctionnement du SVM

## 1. Transformation des données

Chaque audio est transformé en un vecteur de caractéristiques numériques :

- différence des MFCC  
- norme de distance  
- moyenne des écarts  
- similarité cosinus  


## 2. Construction de l’hyperplan

Le SVM cherche à :

- séparer les classes (correct / incorrect)  
- maximiser la marge entre les classes  
- trouver la frontière de décision la plus robuste  


## 3. Décision finale

Pour un nouvel audio :

- le modèle calcule sa position dans l’espace des features  
- il détermine de quel côté de l’hyperplan il se trouve  

Le résultat est :

- correct  
- incorrect  
- probabilité (confidence)  


# Feature Engineering

Le SVM ne peut pas traiter directement les signaux audio.

Il est donc nécessaire de transformer l’audio en variables numériques appelées features.


## Features utilisées

- moyenne des différences MFCC  
- écart-type des différences  
- norme L2 (distance globale)  
- similarité cosinus  

## Objectif des features

Ces caractéristiques représentent le niveau de similarité entre deux audios.


# Principe du système

Pour un audio utilisateur :

1. Chargement de l’audio  
2. Extraction du MFCC  
3. Comparaison avec un audio de référence  
4. Calcul des différences  
5. Création du vecteur de features  
6. Passage au modèle SVM  
7. Classification finale  


# Rôle du SVM dans ce projet

Le SVM sert à :

- apprendre la frontière entre prononciation correcte et incorrecte  
- classer un nouvel audio selon sa similarité avec une référence  


# Rôle de la similarité

Même avec le SVM, des mesures de distance sont utilisées.

Interprétation :

- forte similarité → correct  
- faible similarité → incorrect  


# Limites du SVM

- dépend fortement de la qualité des features  
- ne comprend pas les phonèmes  
- sensible aux variations de voix et d’accent  
- nécessite un dataset équilibré  
- ne traite pas directement le signal audio  


# Résumé du pipeline

Audio utilisateur + Audio de référence  
Extraction MFCC  
Calcul des features  
SVM (classification)  
Correct / Incorrect  
Score de confiance  


# Conclusion

Dans ce système :

- le MFCC transforme l’audio en représentation numérique  
- les features mesurent la similarité entre les signaux  
- le SVM apprend une frontière de décision  
- la classification finale dépend de cette séparation  


Install

In [22]:
!pip install librosa datasets scikit-learn soundfile

Imports

In [23]:
import numpy as np
import librosa
import random

from datasets import load_dataset
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

charger le dataset

In [24]:
dataset = load_dataset(
    "MohamedRashad/common-voice-18-arabic",
    split="train"
)

Extraction MFCC

In [25]:
SR = 16000
N_MFCC = 40
MAX_LEN = 200

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC).T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

Feature Engineering

In [26]:
def extract_features(audio1, audio2):
    mfcc1 = extract_mfcc(audio1).flatten().reshape(1, -1)
    mfcc2 = extract_mfcc(audio2).flatten().reshape(1, -1)

    diff = np.abs(mfcc1 - mfcc2)

    cos = cosine_similarity(mfcc1, mfcc2)[0][0]

    return np.array([
        np.mean(diff),
        np.std(diff),
        np.linalg.norm(diff),
        cos
    ])

Build dataset

In [40]:
X = []
y = []

samples = list(dataset.select(range(1200)))

for i in range(len(samples)):
    try:
        s1 = samples[i]
        a1 = s1["audio"]["array"]

        # ❌ DIFFERENT sentence → class 0
        s2 = random.choice(samples)
        a2 = s2["audio"]["array"]

        X.append(extract_features(a1, a2))
        y.append(0)

        # ✔ FORCE positive sample → class 1
        X.append(extract_features(a1, a1))
        y.append(1)

    except:
        continue

X = np.array(X)
y = np.array(y)

print("Classes:", np.unique(y))

Classes: [0 1]


Normalisation

In [41]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Train SVM

In [42]:
model = SVC(
    kernel='rbf',
    probability=True,
    class_weight='balanced'
)

model.fit(X, y)

SVC(class_weight='balanced', probability=True)

TEST

In [43]:
sample = random.choice(list(dataset))

print("🎤 Read this sentence:")
print("👉", sample["sentence"])

ref_audio = sample["audio"]["array"]

🎤 Read this sentence:
👉 الَّذِي جَعَلَ مَعَ اللَّهِ إِلَهًا آخَرَ فَأَلْقِيَاهُ فِي الْعَذَابِ الشَّدِيدِ


Upload audio

In [44]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

Saving Enregistrement (3).m4a to Enregistrement (3) (1).m4a


Prediction

In [45]:
mfcc1 = extract_mfcc(user_audio).flatten().reshape(1, -1)
mfcc2 = extract_mfcc(ref_audio).flatten().reshape(1, -1)

score = cosine_similarity(mfcc1, mfcc2)[0][0]

print("Similarity:", score)

Similarity: 0.86894304


Resultat

In [46]:
if score > 0.8:
    print("✔ Correct pronunciation")
elif score > 0.6:
    print("⚠️ Acceptable")
else:
    print("❌ Incorrect pronunciation")

✔ Correct pronunciation


# Comparaison des méthodes

## Base commune : MFCC

Toutes les méthodes utilisent le MFCC (Mel-Frequency Cepstral Coefficients).

Il transforme l’audio en matrice numérique représentant :
- le temps  
- les fréquences  
- les caractéristiques de la parole  

# 1. CNN + KNN

## Fonctionnement
1. MFCC → matrice 2D  
2. CNN → extraction de features profondes  
3. Embedding vectoriel  
4. KNN → recherche des voisins les plus proches  

## Avantages
- Très puissant pour les données audio  
- Apprend des patterns complexes  
- Bonne performance sur similarité vocale  

## Inconvénients
- Plus complexe à implémenter  
- Besoin de plus de données  
- Plus lent à entraîner  

---

# 2. Random Forest

## Fonctionnement
1. Extraction de features (MFCC + statistiques)  
2. Plusieurs arbres apprennent des règles  
3. Vote final → correct / incorrect  

## Avantages
- Simple à utiliser  
- Robuste aux données bruitées  
- Bonne interprétabilité  

## Inconvénients
- Ne comprend pas la structure temporelle de l’audio  
- Dépend fortement des features  
- Moins performant sur signal audio brut  

---

# 3. Linear Regression

## Fonctionnement
1. Extraction de features  
2. Apprentissage d’une relation linéaire  
3. Sortie d’un score (0 → 1)  

## Avantages
- Très simple  
- Rapide  
- Facile à interpréter  

## Inconvénients
- Trop simple pour la parole  
- Mauvaise précision  
- Ne capture pas la complexité audio  
- Résultats instables  

---

# SVM (Support Vector Machine)

## Fonctionnement
1. Extraction de features  
2. Projection dans un espace de séparation  
3. Hyperplan optimal  
4. Classification  

## Avantages
- Bon pour classification binaire  
- Efficace sur petits datasets  
- Bonne précision avec features bien choisies  

## Inconvénients
- Sensible aux features  
- Ne traite pas directement l’audio  
- Peut mal généraliser si dataset déséquilibré  


# COMPARAISON GLOBALE

| Méthode              | Performance audio | Complexité | Robustesse | Interprétabilité |
|---------------------|------------------|------------|-------------|------------------|
| CNN + KNN           | ⭐⭐⭐⭐⭐            | Élevée     | ⭐⭐⭐⭐⭐      | Moyenne          |
| Random Forest       | ⭐⭐⭐              | Moyenne    | ⭐⭐⭐⭐       | Élevée           |
| Linear Regression   | ⭐⭐               | Faible     | ⭐⭐         | Très élevée      |
| SVM                 | ⭐⭐⭐⭐             | Moyenne    | ⭐⭐⭐⭐       | Moyenne          |

# ANALYSE GLOBALE

## Meilleure méthode : CNN + KNN
- Capture les patterns audio complexes  
- Meilleure pour similarité vocale  
- Plus proche des systèmes industriels  

## SVM
- Bon compromis précision / simplicité  
- Fonctionne bien si features bien conçues  

## Random Forest
- Stable et robuste  
- Mais limité pour données audio complexes  

## Linear Regression
- Simple baseline  
- Pas adapté à la compréhension de la parole  

# **Reférence :**

F. Nazir, M. N. Majeed, M. A. Ghazanfar, and M. Maqsood,
“Mispronunciation Detection Using Deep Convolutional Neural Network Features and Transfer Learning-Based Model for Arabic Phonemes,”
IEEE Access, vol. 7, pp. 52589–52608, 2019.
Available: https://www.academia.edu/165325079/Mispronunciation_Detection_Using_Deep_Convolutional_Neural_Network_Features_and_Transfer_Learning_Based_Model_for_Arabic_Phonemes

A.Maghraby, A.Mahjoob , W.Alsaedi , A.Alsaedi , M.Alsaedi , A.Alotaibi ,  "Enhancing Arabic Reading Proficiency through Artificial Intelligent Application", Life Sci J 2024;21(5):39-4
9]. ISSN 1097-8135 (print); ISSN 2372-613X (online). Available : https://www.lifesciencesite.com/lsj/lsj210524/03_38991lsj210524_26_32.pdf